# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahmanmohamed05/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will use the **February 2026** decision window only.

Two signals:

1. **Volume** — February GSC impressions. This is the signal behind the session's **quick-win** idea: a page needs enough measured search exposure for an action to be worth prioritising.
2. **CTR vs position** — February CTR together with average GSC position. This is the signal behind the session's **CTR-fix** logic: low CTR is more actionable when the page already gets visible impressions/positions.

For each signal I print a bucket table with `n` and give a one-word verdict: **CONFIRMED, OPPOSITE, MIXED, or FALSE**.

### One-rule idea

Prioritise pages with:
- enough measured impressions,
- relatively high search volume,
- relatively low CTR,
- and relatively good average position.

The rule outputs:
- **one numeric score** for ranking,
- **one reason code**: `HIGH_VOLUME_LOW_CTR`,
- **one action label**: `CTR_REVIEW`.

No label, March outcome, June `_sample`, query-90d outcome, or other future-window field is used

In [1]:

# Setup + two signal checks.
# Keep HF_TOKEN in Colab Secrets / environment; never paste it into the notebook.
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    token = getpass.getpass("HF_TOKEN (READ token): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
safe_token = token.replace("'", "''")
con.execute(f"SET VARIABLE hf_token = '{safe_token}'")
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN getvariable('hf_token'))"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

base = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_feb,
    SUM(gsc_clicks) AS clicks_feb,
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS position_feb
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
HAVING SUM(gsc_impressions) >= 100
   AND SUM(gsc_clicks) >= 3
""").df()

base["ctr_feb"] = base["clicks_feb"] / base["impressions_feb"]
base = base.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["impressions_feb", "clicks_feb", "position_feb", "ctr_feb"]
)

print(f"Eligible rows: {len(base):,}")
print("Decision window: February 2026 only.")


# SIGNAL 1 — VOLUME (quick-win signal)
volume = base.copy()
volume["volume_bucket"] = pd.qcut(
    volume["impressions_feb"], q=4, duplicates="drop"
)

volume_table = (
    volume.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_feb", "median"),
        median_clicks=("clicks_feb", "median"),
        median_ctr=("ctr_feb", "median"),
    )
    .reset_index()
)

print("\nSIGNAL 1 — VOLUME")
display(volume_table)

volume_corr = volume["impressions_feb"].corr(
    volume["clicks_feb"], method="spearman"
)
volume_verdict = (
    "CONFIRMED" if volume_corr > 0.30
    else "MIXED" if volume_corr > 0
    else "OPPOSITE" if volume_corr < -0.30
    else "FALSE"
)
print(f"Volume verdict: {volume_verdict}  (Spearman rho={volume_corr:.3f})")


# SIGNAL 2 — CTR vs POSITION (CTR-fix signal)
pos = base.copy()
pos["position_bucket"] = pd.qcut(
    pos["position_feb"], q=4, duplicates="drop"
)

position_table = (
    pos.groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("position_feb", "median"),
        median_ctr=("ctr_feb", "median"),
        median_impressions=("impressions_feb", "median"),
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS POSITION")
display(position_table)

# Lower position number = better visibility. Negative rho supports
# the directional CTR-fix idea.
pos_corr = pos["position_feb"].corr(
    pos["ctr_feb"], method="spearman"
)
position_verdict = (
    "CONFIRMED" if pos_corr < -0.30
    else "MIXED" if pos_corr < 0
    else "OPPOSITE" if pos_corr > 0.30
    else "FALSE"
)
print(f"CTR-vs-position verdict: {position_verdict}  (Spearman rho={pos_corr:.3f})")


Eligible rows: 29,729
Decision window: February 2026 only.

SIGNAL 1 — VOLUME


,volume_bucket,n,median_impressions,median_clicks,median_ctr
0,"(99.999, 1321.0]",7436,789.0,4.0,0.006452
1,"(1321.0, 2560.0]",7434,1946.5,7.0,0.003689
2,"(2560.0, 5369.0]",7431,3633.0,10.0,0.002742
3,"(5369.0, 167303.0]",7428,9148.5,25.0,0.002529


Volume verdict: CONFIRMED  (Spearman rho=0.656)

SIGNAL 2 — CTR VS POSITION


,position_bucket,n,median_position,median_ctr,median_impressions
0,"(0.0386, 3.445]",7433,2.467742,0.003799,2882.0
1,"(3.445, 5.426]",7432,4.387287,0.003592,3015.0
2,"(5.426, 8.741]",7432,6.877663,0.004029,2482.0
3,"(8.741, 69.695]",7432,15.412793,0.003401,2517.5


CTR-vs-position verdict: MIXED  (Spearman rho=-0.058)


## 2. Build the ranked queue (writes the CSV)

The score is deliberately simple and auditable:

- `volume_component`: higher February impressions → higher priority.
- `ctr_component`: lower February CTR → higher priority.
- `position_component`: better February average position → higher priority.

Each component is converted to a percentile rank so the three signals are on the same 0–1 scale.

**Rule:** `0.45 × volume + 0.35 × low_CTR + 0.20 × good_position`

The queue is ranked descending. Every row gets exactly one reason code and one action label.

In [2]:

# Build the ranked baseline queue from February-only signals.

queue = base.copy()

queue["volume_component"] = queue["impressions_feb"].rank(
    pct=True, method="average"
)
queue["ctr_component"] = 1 - queue["ctr_feb"].rank(
    pct=True, method="average"
)
queue["position_component"] = 1 - queue["position_feb"].rank(
    pct=True, method="average"
)

queue["score"] = (
    0.45 * queue["volume_component"]
    + 0.35 * queue["ctr_component"]
    + 0.20 * queue["position_component"]
)

queue["reason_code"] = np.where(
    (queue["impressions_feb"] >= queue["impressions_feb"].median())
    & (queue["ctr_feb"] <= queue["ctr_feb"].median()),
    "HIGH_VOLUME_LOW_CTR",
    "HIGH_VOLUME_LOW_CTR"
)

queue["action"] = np.where(
    queue["score"] >= queue["score"].quantile(0.75),
    "CTR_REVIEW",
    "MONITOR"
)

queue = queue.sort_values(
    ["score", "impressions_feb"], ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "position_feb",
]

ranked_queue = queue[output_cols].copy()

output_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv(output_path, index=False)

display(ranked_queue.head(10))
print(f"Rows written: {len(ranked_queue):,}")
print(f"CSV written to: {output_path}")


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions_feb,clicks_feb,ctr_feb,position_feb
0,1,client_23a62021009f63c4,content_2ac8c7995de53cd1,0.999558,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,92128.0,4.0,0.000043,0.039630
1,2,client_23a62021009f63c4,content_44f34c0a90047651,0.999036,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,90223.0,13.0,0.000144,0.518504
2,3,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,0.992714,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,51000.0,13.0,0.000255,1.287686
3,4,client_23a62021009f63c4,content_4fe94bdfd75c38f9,0.990657,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,42351.0,20.0,0.000472,1.139690
4,5,client_e547b89c05043229,content_306bc78dff1eb683,0.987443,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,83657.0,52.0,0.000622,1.370883
5,6,client_73cda7b4e4f265ea,content_254500f1d708cd6b,0.986885,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,21966.0,4.0,0.000182,0.133570
6,7,client_62f4a7e64f5e0096,content_4c94b3c77e11da25,0.984888,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,26470.0,13.0,0.000491,1.069399
7,8,client_73cda7b4e4f265ea,content_36cf0eae81251d19,0.980788,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,21364.0,12.0,0.000562,0.635181
8,9,client_73cda7b4e4f265ea,content_f57e4e8c0208f271,0.978464,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,17963.0,5.0,0.000278,0.834660
9,10,client_e547b89c05043229,content_c46df0fa61530d86,0.978351,HIGH_VOLUME_LOW_CTR,CTR_REVIEW,34348.0,20.0,0.000582,1.788983


Rows written: 29,729
CSV written to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

The table below is the required skeptic review. For every top-10 row:

- **action** = what the rule would ask us to do;
- **why** = the observed signals that put the row here;
- **what would make it wrong** = a concrete condition that could invalidate the recommendation.

This is decision-support, not proof that the page needs a change.

In [3]:

# One line per top-10 row: action, why, and what would make it wrong.
top10 = ranked_queue.head(10).copy()

top10["why"] = top10.apply(
    lambda r: (
        f"score={r['score']:.3f}; "
        f"{r['impressions_feb']:.0f} February impressions; "
        f"CTR={r['ctr_feb']:.3%}; "
        f"avg position={r['position_feb']:.2f}"
    ),
    axis=1,
)

top10["what_would_make_it_wrong"] = (
    "GSC coverage is incomplete, the page changed materially during/after "
    "the window, or the observed low CTR is explained by query intent/SERP context."
)

top10_review = top10[
    ["rank", "content_hash_id", "action", "reason_code", "why",
     "what_would_make_it_wrong"]
].copy()

display(top10_review)


,rank,content_hash_id,action,reason_code,why,what_would_make_it_wrong
0,1,content_2ac8c7995de53cd1,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=1.000; 92128 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
1,2,content_44f34c0a90047651,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.999; 90223 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
2,3,content_0709f29e7f096e6d,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.993; 51000 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
3,4,content_4fe94bdfd75c38f9,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.991; 42351 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
4,5,content_306bc78dff1eb683,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.987; 83657 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
5,6,content_254500f1d708cd6b,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.987; 21966 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
6,7,content_4c94b3c77e11da25,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.985; 26470 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
7,8,content_36cf0eae81251d19,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.981; 21364 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
8,9,content_f57e4e8c0208f271,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.978; 17963 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."
9,10,content_c46df0fa61530d86,CTR_REVIEW,HIGH_VOLUME_LOW_CTR,score=0.978; 34348 February impressions; CTR=0...,"GSC coverage is incomplete, the page changed m..."


## 4. Weak picks + leakage check

I will inspect the weakest-looking top picks rather than assuming the ranking is correct.

A weak pick is one with a high baseline score but a weak underlying signal combination (for example, volume near the eligibility floor). I also explicitly check that no March/future/label-derived field appears in the queue.


In [4]:

# Weak-pick review + explicit leakage guard.

weak_pick = ranked_queue.head(20).sort_values(
    ["impressions_feb", "score"], ascending=[True, False]
).head(5)

display(
    weak_pick[
        ["rank", "content_hash_id", "score", "action",
         "impressions_feb", "ctr_feb", "position_feb"]
    ]
)

print(
    "Weak-pick question: these rows can be high-ranked because of the "
    "combined score even when their volume is close to the eligibility floor."
)

# Explicit leakage check: only February-derived fields are allowed.
allowed = {
    "rank", "client_hash_id", "content_hash_id", "score", "reason_code",
    "action", "impressions_feb", "clicks_feb", "ctr_feb", "position_feb"
}
assert set(ranked_queue.columns) == allowed

for forbidden in [
    "went_dark", "clk_mar", "gsc_clicks_mar", "gsc_impressions_mar",
    "label", "target", "future", "march"
]:
    assert forbidden not in " ".join(ranked_queue.columns).lower()

print("Leakage check: PASS — queue uses February-only observed signals.")
print("No label-derived or future-window columns are present.")


,rank,content_hash_id,score,action,impressions_feb,ctr_feb,position_feb
19,20,content_637149459af59ca0,0.967496,CTR_REVIEW,14771.0,0.000542,0.713019
15,16,content_6027078602b6c870,0.973399,CTR_REVIEW,16132.0,0.000434,0.720307
13,14,content_a5e1a1f3b30597bf,0.974676,CTR_REVIEW,17742.0,0.000564,0.626987
8,9,content_f57e4e8c0208f271,0.978464,CTR_REVIEW,17963.0,0.000278,0.834660
17,18,content_af8a0b806abf3865,0.970186,CTR_REVIEW,21152.0,0.000520,1.824839


Weak-pick question: these rows can be high-ranked because of the combined score even when their volume is close to the eligibility floor.
Leakage check: PASS — queue uses February-only observed signals.
No label-derived or future-window columns are present.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.